# 02: GRANT, REVOKE

**Exam objective:** Configure access controls using the UI and SQL by applying 
GRANT and REVOKE privileges to principals (users, groups, and service 
principals) at appropriate levels of the security hierarchy.

**Setup:** Group `analysts` created in workspace identity settings. No members 
added — empty group is sufficient as a grant target.

In [0]:
SHOW GROUPS;

In [0]:
USE CATALOG certprep;
USE SCHEMA governance;

-- Create a table the analysts group will be granted access to
CREATE TABLE IF NOT EXISTS sales_data (
  region STRING,
  amount DECIMAL(10,2),
  customer_email STRING,
  sale_date DATE
);

INSERT INTO sales_data VALUES
  ('North', 1250.00, 'alice@example.com', '2026-01-15'),
  ('South', 890.50, 'bob@example.com', '2026-01-16'),
  ('East', 2100.75, 'carol@example.com', '2026-01-17'),
  ('West', 1575.25, 'dan@example.com', '2026-01-18');

## SHOW GRANTS
- SHOW GRANTS only displays grants explicitly issued via GRANT statements. 
- On initial run, the 3 statements below return no rows, as implicit privileges through ownership are not displayed.
- I am the owner of the table, schema, and catalog, thus I have the implict privileges.
- View the owner of each object via DESCRIBE:
    - DESCRIBE CATALOG *catalog_name*;
    - DESCRIBE SCHEMA *catalog_name*.*schema_name*;
    - DESCRIBE TABLE EXTENDED *catalog_name*.*schema_name*.*table_name*;

In [0]:
-- See what privileges currently exist on the table
SHOW GRANTS ON TABLE sales_data;

In [0]:
-- See what privileges currently exist on the schema
SHOW GRANTS ON SCHEMA governance;

In [0]:
-- See what privileges currently exist on the catalog
SHOW GRANTS ON CATALOG certprep;

## GRANT SELECT

In [0]:
-- Grant SELECT on the table to the analysts group
GRANT SELECT ON TABLE sales_data TO analysts;

In [0]:
-- Verfiy grant
SHOW GRANTS ON TABLE sales_data;

### Question
If a member of the analysts group tried to run `SELECT * FROM certprep.governance.sales_data` right now, would they succeed? Why or why not?

Answer: No. While the analysts group has been granted permissions on the table `sales_data`, they have not been granted permissions on the catalog and schema levels. Catalog, schema, and table grants are required in order to interact with the table object. This is known as the privilege-chain rule. 

## GRANT USE

In [0]:
-- Grant analyst group USE SCHEMA
GRANT USE SCHEMA ON SCHEMA governance TO analysts;

-- Grant analyst group USE CATALOG
GRANT USE CATALOG ON CATALOG certprep TO analysts;

In [0]:
-- Now check all three levels
SHOW GRANTS ON CATALOG certprep;

In [0]:
SHOW GRANTS ON SCHEMA governance;

In [0]:
SHOW GRANTS ON TABLE sales_data;

## REVOKE

In [0]:
-- Revoke SELECT
REVOKE SELECT ON TABLE sales_data FROM analysts;

In [0]:
SHOW GRANTS ON TABLE sales_data;

### Note on REVOKE
- REVOKE removes a previously-granted privilege.
- REVOKE is *not* a prohibition. Lacking a privilege is one thing, being preveneted from receiving the privilege is another.

## DENY in Unity Catalog VS Hive Metastore

DENY is not supported by Unity Catalog. Attempting `DENY SELECT ON TABLE 
sales_data TO analysts` produces:

`[UC_COMMAND_NOT_SUPPORTED.WITHOUT_RECOMMENDATION] The command(s): DENY are 
not supported in Unity Catalog. SQLSTATE: 0AKUC`

DENY existed in the legacy Hive metastore (the `hive_metastore` catalog) to 
override broad GRANTs. UC takes a different approach: the default is no 
access, and access only exists through an explicit grant chain. There is 
nothing to override, so DENY is unnecessary.

The UC pattern for "this group should not be able to access this specific 
table" is:
1. Don't grant SELECT on the table to that group.
2. Make sure no higher-level grant (e.g., SELECT on the schema or catalog) 
   cascades down to it.

If a broader grant exists that would cascade, you must REVOKE that broader 
grant and re-grant more selectively. UC doesn't offer a way to surgically 
deny one table while keeping a schema-level grant active.

## Clean Up

In [0]:
-- Reset state so the next exercise starts clean
REVOKE SELECT ON SCHEMA governance FROM analysts;
REVOKE USE SCHEMA ON SCHEMA governance FROM analysts;
REVOKE USE CATALOG ON CATALOG certprep FROM analysts;

## Self Check Questions
1. A user is in two groups: analysts (which has SELECT on a table) and contractors (which has no grants on the table). Can the user query the table? What if contractors had USE CATALOG and USE SCHEMA but not SELECT on the table — does that change anything?
2. You grant a user USE CATALOG on prod and SELECT on prod.finance.revenue, but they get an access-denied error querying the table. What did you forget?
3. What's the difference between granting SELECT on a schema versus granting SELECT on each table in the schema individually? Are they functionally equivalent?
4. You are the owner of a table. A colleague is also added as a co-owner via ALTER TABLE … OWNER TO. Can the original owner still drop the table? (Trick question — look up table ownership rules.)
5. Why does the privilege hierarchy use USE CATALOG and USE SCHEMA rather than just inheriting access automatically from grants at higher levels?

## Self Check Answers
1. The user would be able to run a SELECT on the table via the analysts group's grants. Even if contractors have USE CATALOG and USE SCHEMA, the group still lacks the privilege to SELECT on that table. Since the user is a member of both groups, they are able to query SELECT through the analysts permissions only, not the contractors permissions. It is also worth noting that other members of contractors who are not also in analysts would still not be able to SELECT from the table.
2. The user also requires USE SCHEMA to be granted on prod.finance. Without it, the privilege chain is broken between the catalog and table levels by the omission of the schema level privileges.
3. Granting SELECT on a schema cascades to all tables presently in the schema, as well as tables created in the future. Manually granting SELECT for each table in the schema only contains the singular table in its scope, and new tables would likewise require the granular granting again. The key difference functionally, granting SELECT on a schema cascades a user's/group's permissions to the tables or objects within it, while granting SELECT on a table impacts only the user's/group's permissions on that single table and does not cascade to other objects.
4. Co-ownership is a misconception. There can only be one owner of a table at a time, thus `ALTER TABLE ... OWNER TO` actually transfers ownership instead of extending it.  The original owner in this scenario would no longer be the owner and cannot drop the table unless they have other privileges. The ex-owner can drop the table if they are the owner of the parent catalog or schema, have MANAGE privilege on the table (possibly inherited), or metastore admin status. Without any of those, they can no longer drop the table.
5. Least access principle reduces the chance of unauthorized or unapproved access. For example, User A needs access to schema_a and User B needs access to schema_b within the same catalog, but neither should have access to the other respective schema. If USE CATALOG cascaded as USE SCHEMA on each schema, both users would have access they should not have. Instead, USE CATALOG allows traversal of the catalog namespace, but does not blanket access to its contents. By explicitly requiring both catalog and schema privileges, governance can manage access more precisely instead of broadly. 